# UROP Research Log — May 29, 2026

## Objective
Exploratory Data Analysis (EDA) of Cboe SPX options tick data (2022).
Understand the structure, quality, and key columns before building
the Stage 2 synchronization pipeline.

## Why
Before writing any Stage 2 code, I need to understand what the raw data
actually looks like. Key questions:
- What columns are available and what do they mean?
- How many trades per day? What's the data volume?
- Are there data quality issues (nulls, canceled trades, outliers)?
- Can I identify the exact SPX ATM call option at 3:55 PM ET?
- What does the roll date data look like (3rd Friday of each month)?

## Data Source
- Cboe SPX Options Tick Data (2022): Purchased from Cboe Datashop
  251 daily CSV files, ~14.8GB total
  Located at: ../data/raw/cboe_spx_2022/

## Approach
1. Load one day of data and inspect structure
2. Check column types, nulls, value ranges
3. Filter to ATM call options only
4. Inspect a roll date (3rd Friday) in detail
5. Define quality filter criteria

## Notes
- Data too large for GitHub (14.8GB) — stored locally only
- Roll dates in 2022: Jan 21, Feb 18, Mar 18, Apr 14, May 20,
  Jun 17, Jul 15, Aug 19, Sep 16, Oct 21, Nov 18, Dec 16

In [1]:
# -- Load Single Day & Inspect Structure --
import pandas as pd
import numpy as np
import os

# Load one day (Jan 3, 2022)
path = "../data/raw/cboe_spx_2022/UnderlyingOptionsTradesCalcs_2022-01-03.csv"
df = pd.read_csv(path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)
print("\nFirst 3 rows:")
print(df.head(3))

Shape: (186792, 23)

Columns:
['underlying_symbol', 'market_date', 'quote_datetime', 'sequence_number', 'root', 'expiration', 'strike', 'option_type', 'exchange_id', 'trade_size', 'trade_price', 'trade_condition_id', 'canceled_trade_condition_id', 'best_bid', 'best_ask', 'trade_iv', 'trade_delta', 'trade_gamma', 'trade_vega', 'trade_theta', 'trade_rho', 'underlying_bid', 'underlying_ask']

Dtypes:
underlying_symbol                  str
market_date                        str
quote_datetime                     str
sequence_number                  int64
root                               str
expiration                         str
strike                         float64
option_type                        str
exchange_id                      int64
trade_size                       int64
trade_price                    float64
trade_condition_id               int64
canceled_trade_condition_id      int64
best_bid                       float64
best_ask                       float64
trade_iv      

In [2]:
# -- Basic Data Quality Check --

print("=== Null Counts ===")
print(df.isnull().sum())

print("\n=== Canceled Trades ===")
print(f"canceled_trade_condition_id > 0: {(df['canceled_trade_condition_id'] > 0).sum()}")
print(f"Total trades: {len(df)}")

print("\n=== Option Type Distribution ===")
print(df['option_type'].value_counts())

print("\n=== Root Distribution (SPX vs SPXW) ===")
print(df['root'].value_counts())

print("\n=== quote_datetime range ===")
df['quote_datetime'] = pd.to_datetime(df['quote_datetime'])
print(f"Min: {df['quote_datetime'].min()}")
print(f"Max: {df['quote_datetime'].max()}")

=== Null Counts ===
underlying_symbol              0
market_date                    0
quote_datetime                 0
sequence_number                0
root                           0
expiration                     0
strike                         0
option_type                    0
exchange_id                    0
trade_size                     0
trade_price                    0
trade_condition_id             0
canceled_trade_condition_id    0
best_bid                       0
best_ask                       0
trade_iv                       0
trade_delta                    0
trade_gamma                    0
trade_vega                     0
trade_theta                    0
trade_rho                      0
underlying_bid                 0
underlying_ask                 0
dtype: int64

=== Canceled Trades ===
canceled_trade_condition_id > 0: 0
Total trades: 186792

=== Option Type Distribution ===
option_type
P    103341
C     83451
Name: count, dtype: int64

=== Root Distribution (SPX vs 

### Thought Process — EDA Observations (Jan 3, 2022)

**Data Quality**: No nulls, no canceled trades on this day. Clean.

**Option Type**: More Puts (103k) than Calls (83k) — typical for SPX
where investors buy puts for downside protection.

**Root**: SPXW (172k) >> SPX (14k)
- SPX = standard monthly options (3rd Friday expiry)
- SPXW = weekly options (every Friday expiry)
- For BXM replication, we only need SPX root, not SPXW
- This means we can immediately filter out ~92% of data

**Timestamp**: quote_datetime starts from 2022-01-02 20:15 (previous evening)
and goes to 2022-01-03 16:16. Overnight/pre-market trades exist.
For our analysis, we only need regular session: 09:30 ~ 16:00 ET.

**Open Question**: Should we filter by trade_condition_id as well?
The proposal mentions excluding codes A-H, f-t. Need to check
what values actually appear in this column.

In [3]:
# -- Filter to BXM-Relevant Options --

# 1. Regular session only (9:30 AM - 4:15 PM ET)
df_filtered = df[
    (df['quote_datetime'].dt.time >= pd.Timestamp('09:30').time()) &
    (df['quote_datetime'].dt.time <= pd.Timestamp('16:15').time())
].copy()

# 2. SPX root only (monthly options, not weekly SPXW)
df_filtered = df_filtered[df_filtered['root'] == 'SPX']

# 3. Call options only (BXM sells calls)
df_filtered = df_filtered[df_filtered['option_type'] == 'C']

print(f"Original rows    : {len(df):,}")
print(f"After filtering  : {len(df_filtered):,}")
print(f"Reduction        : {(1 - len(df_filtered)/len(df))*100:.1f}%")

print("\n=== trade_condition_id values ===")
print(df_filtered['trade_condition_id'].value_counts())

print("\n=== Expiration dates available ===")
print(df_filtered['expiration'].value_counts())

Original rows    : 186,792
After filtering  : 4,917
Reduction        : 97.4%

=== trade_condition_id values ===
trade_condition_id
119    2337
18     1161
133     753
13      233
126     167
120     118
40       56
114      54
118      19
41       14
123       4
95        1
Name: count, dtype: int64

=== Expiration dates available ===
expiration
2022-01-21    1499
2022-03-18    1307
2022-02-18     852
2022-12-16     268
2022-04-14     180
2022-06-17     180
2022-08-19     154
2023-12-15      97
2022-05-20      90
2022-07-15      85
2022-09-16      75
2023-06-16      51
2022-11-18      32
2023-01-20      20
2022-10-21       9
2023-03-17       7
2024-12-20       7
2025-12-19       4
Name: count, dtype: int64


### Thought Process — Filtering Observations

**97.4% reduction**: From 186k → 4,917 rows after filtering to
SPX monthly calls during regular session. Very manageable I love it!

**trade_condition_id**: Multiple numeric codes present (18, 119, 133, etc.)
The proposal mentions excluding codes A-H, f-t — but these appear to be
numeric, not letter codes. Need to clarify with Professor Dodson
which numeric codes correspond to delayed or canceled trades.
Code 18 appears most frequently — worth investigating.

**Expiration dates**: Jan 21 (1,499) and Mar 18 (1,307) dominate.
On Jan 3, traders are most active in near-term expiry
(Jan 21 = 3rd Friday of January = BXM roll date). Expected.

**For BXM replication on Jan 3**:
- Target expiry = 2022-01-21 (next 3rd Friday)
- 11:00 AM ET: ATM strike = closest strike at or above SPX level
- 11:30 AM ~ 1:30 PM ET: VWAP calculation window (per official Cboe methodology)

In [4]:
# -- ATM Strike Selection at 11:00 AM ET --

# Filter to Jan 21 expiry (next roll date from Jan 3)
df_jan21 = df_filtered[df_filtered['expiration'] == '2022-01-21'].copy()

# Get SPX level just before 11:00 AM ET
window_11am = df_jan21[
    (df_jan21['quote_datetime'].dt.time >= pd.Timestamp('10:50').time()) &
    (df_jan21['quote_datetime'].dt.time <= pd.Timestamp('11:00').time())
]

spx_level = window_11am['underlying_bid'].mean()
print(f"SPX level ~11:00 AM ET: {spx_level:.2f}")

# ATM Strike = closest strike at or above SPX level
available_strikes = sorted(df_jan21['strike'].unique())
atm_strike = min([s for s in available_strikes if s >= spx_level])
print(f"Available strikes near ATM: {[s for s in available_strikes if abs(s - spx_level) < 100]}")
print(f"Selected ATM Strike: {atm_strike}")

SPX level ~11:00 AM ET: 4778.08
Available strikes near ATM: [np.float64(4680.0), np.float64(4690.0), np.float64(4695.0), np.float64(4700.0), np.float64(4705.0), np.float64(4710.0), np.float64(4715.0), np.float64(4720.0), np.float64(4725.0), np.float64(4730.0), np.float64(4735.0), np.float64(4740.0), np.float64(4750.0), np.float64(4755.0), np.float64(4760.0), np.float64(4765.0), np.float64(4770.0), np.float64(4775.0), np.float64(4780.0), np.float64(4785.0), np.float64(4790.0), np.float64(4795.0), np.float64(4800.0), np.float64(4805.0), np.float64(4810.0), np.float64(4815.0), np.float64(4820.0), np.float64(4825.0), np.float64(4830.0), np.float64(4835.0), np.float64(4840.0), np.float64(4845.0), np.float64(4850.0), np.float64(4855.0), np.float64(4860.0), np.float64(4865.0), np.float64(4870.0), np.float64(4875.0)]
Selected ATM Strike: 4780.0


In [5]:
# -- VWAP Calculation (11:30 AM ~ 1:30 PM ET) --

# Filter to ATM strike + VWAP window
df_vwap = df_jan21[
    (df_jan21['strike'] == atm_strike) &
    (df_jan21['quote_datetime'].dt.time >= pd.Timestamp('11:30').time()) &
    (df_jan21['quote_datetime'].dt.time <= pd.Timestamp('13:30').time())
].copy()

print(f"Trades in VWAP window: {len(df_vwap)}")
print(f"Total volume: {df_vwap['trade_size'].sum():,}")

if len(df_vwap) > 0:
    vwap = (df_vwap['trade_price'] * df_vwap['trade_size']).sum() / df_vwap['trade_size'].sum()
    print(f"\nVWAP (option sell price): ${vwap:.4f}")
    print(f"Min price: ${df_vwap['trade_price'].min():.4f}")
    print(f"Max price: ${df_vwap['trade_price'].max():.4f}")
    print(f"\nSample trades:")
    print(df_vwap[['quote_datetime', 'strike', 'trade_price', 'trade_size']].head(5))
else:
    print("No trades found in VWAP window!")

Trades in VWAP window: 5
Total volume: 26

VWAP (option sell price): $46.3008
Min price: $42.5700
Max price: $46.7000

Sample trades:
              quote_datetime  strike  trade_price  trade_size
2940 2022-01-03 11:37:18.179  4780.0        42.57           1
2941 2022-01-03 11:37:18.179  4780.0        42.57           1
2942 2022-01-03 12:16:09.321  4780.0        45.22           1
2943 2022-01-03 12:21:06.611  4780.0        46.06           1
2944 2022-01-03 12:29:05.478  4780.0        46.70          22


### Thought Process — VWAP Result on Jan 3 (Non-Roll Date)

VWAP =`$46.30`, but only 5 trades / 26 contracts in the window.
This is statistically unreliable — one large trade (22 contracts at `$46.70`)
dominates the entire VWAP calculation.

**Root cause**: Jan 3 is NOT a roll date (roll date = Jan 21).
On non-roll dates, SPX monthly ATM options are thinly traded.
Heavy trading only happens ON the roll date itself (3rd Friday).

**Next step**: Run the same logic on an actual roll date (Jan 21, 2022)
to see if VWAP window has sufficient volume.
This is the real test of our pipeline.

In [6]:
# -- Load Roll Date (Jan 21, 2022) & Run Full Pipeline --

# Load Jan 21 (first roll date of 2022)
path_roll = "../data/raw/cboe_spx_2022/UnderlyingOptionsTradesCalcs_2022-01-21.csv"
df_roll = pd.read_csv(path_roll)
df_roll['quote_datetime'] = pd.to_datetime(df_roll['quote_datetime'])

# Filter: SPX monthly calls, regular session
df_roll_filtered = df_roll[
    (df_roll['root'] == 'SPX') &
    (df_roll['option_type'] == 'C') &
    (df_roll['quote_datetime'].dt.time >= pd.Timestamp('09:30').time()) &
    (df_roll['quote_datetime'].dt.time <= pd.Timestamp('16:15').time())
].copy()

print(f"Total rows: {len(df_roll):,}")
print(f"After filtering: {len(df_roll_filtered):,}")
print(f"\nExpiration dates:")
print(df_roll_filtered['expiration'].value_counts().head(5))

Total rows: 316,171
After filtering: 13,872

Expiration dates:
expiration
2022-02-18    5227
2022-03-18    3672
2022-04-14     785
2022-06-17     598
2022-09-16     595
Name: count, dtype: int64


In [7]:
# -- ATM Strike Selection & VWAP on Roll Date --

# Target: Feb 18 expiry (new options being sold today)
df_feb18 = df_roll_filtered[df_roll_filtered['expiration'] == '2022-02-18'].copy()

# SPX level just before 11:00 AM ET
window_11am = df_feb18[
    (df_feb18['quote_datetime'].dt.time >= pd.Timestamp('10:50').time()) &
    (df_feb18['quote_datetime'].dt.time <= pd.Timestamp('11:00').time())
]

spx_level = window_11am['underlying_bid'].mean()
print(f"SPX level ~11:00 AM ET: {spx_level:.2f}")

# ATM Strike selection
available_strikes = sorted(df_feb18['strike'].unique())
atm_strike = min([s for s in available_strikes if s >= spx_level])
print(f"Selected ATM Strike: {atm_strike}")

# VWAP calculation (11:30 AM ~ 1:30 PM ET)
df_vwap = df_feb18[
    (df_feb18['strike'] == atm_strike) &
    (df_feb18['quote_datetime'].dt.time >= pd.Timestamp('11:30').time()) &
    (df_feb18['quote_datetime'].dt.time <= pd.Timestamp('13:30').time())
].copy()

print(f"\nTrades in VWAP window: {len(df_vwap)}")
print(f"Total volume: {df_vwap['trade_size'].sum():,}")

if len(df_vwap) > 0:
    vwap = (df_vwap['trade_price'] * df_vwap['trade_size']).sum() / df_vwap['trade_size'].sum()
    print(f"VWAP (option sell price): ${vwap:.4f}")
    print(f"Min price: ${df_vwap['trade_price'].min():.4f}")
    print(f"Max price: ${df_vwap['trade_price'].max():.4f}")

SPX level ~11:00 AM ET: 4478.28
Selected ATM Strike: 4480.0

Trades in VWAP window: 685
Total volume: 3,738
VWAP (option sell price): $92.8499
Min price: $83.9000
Max price: $109.7000


### Thought Process — Roll Date VWAP vs Non-Roll Date

Jan 3 (non-roll): 5 trades, 26 contracts → VWAP unreliable
Jan 21 (roll date): 685 trades, 3,738 contracts → VWAP robust

This confirms the pipeline is working correctly.
Roll dates have dramatically higher liquidity in ATM options,
which is exactly why BXM uses the 11:30 AM ~ 1:30 PM VWAP window
on roll dates specifically.

Jan 21 Roll Date Result:
- SPX level at 11:00 AM ET: 4478.28
- ATM Strike selected: 4480.0
- VWAP (option sell price): `$92.85`
- Price range in window: `$83.90` ~ `$109.70`

In [8]:
# -- All 2022 Roll Dates VWAP Pipeline --
import glob

# 2022 BXM roll dates (3rd Friday of each month)
roll_dates = [
    "2022-01-21", "2022-02-18", "2022-03-18", "2022-04-14",
    "2022-05-20", "2022-06-17", "2022-07-15", "2022-08-19",
    "2022-09-16", "2022-10-21", "2022-11-18", "2022-12-16"
]

results = []

for roll_date in roll_dates:
    path = f"../data/raw/cboe_spx_2022/UnderlyingOptionsTradesCalcs_{roll_date}.csv"
    
    if not os.path.exists(path):
        print(f"{roll_date}: File not found")
        continue
    
    df_r = pd.read_csv(path)
    df_r['quote_datetime'] = pd.to_datetime(df_r['quote_datetime'])
    
    # Filter: SPX monthly calls, regular session
    df_r = df_r[
        (df_r['root'] == 'SPX') &
        (df_r['option_type'] == 'C') &
        (df_r['quote_datetime'].dt.time >= pd.Timestamp('09:30').time()) &
        (df_r['quote_datetime'].dt.time <= pd.Timestamp('16:15').time())
    ].copy()
    
    # Next roll date expiry (new options)
    next_idx = roll_dates.index(roll_date) + 1
    if next_idx >= len(roll_dates):
        next_expiry = "2023-01-20"
    else:
        next_expiry = roll_dates[next_idx]
    
    df_next = df_r[df_r['expiration'] == next_expiry].copy()
    
    if len(df_next) == 0:
        print(f"{roll_date}: No data for expiry {next_expiry}")
        continue
    
    # ATM Strike at 11:00 AM ET
    window_11am = df_next[
        (df_next['quote_datetime'].dt.time >= pd.Timestamp('10:50').time()) &
        (df_next['quote_datetime'].dt.time <= pd.Timestamp('11:00').time())
    ]
    
    if len(window_11am) == 0:
        print(f"{roll_date}: No data near 11:00 AM")
        continue
    
    spx_level = window_11am['underlying_bid'].mean()
    available_strikes = sorted(df_next['strike'].unique())
    valid_strikes = [s for s in available_strikes if s >= spx_level]
    
    if not valid_strikes:
        print(f"{roll_date}: No valid strike found")
        continue
    
    atm_strike = min(valid_strikes)
    
    # VWAP 11:30 AM ~ 1:30 PM ET
    df_vwap = df_next[
        (df_next['strike'] == atm_strike) &
        (df_next['quote_datetime'].dt.time >= pd.Timestamp('11:30').time()) &
        (df_next['quote_datetime'].dt.time <= pd.Timestamp('13:30').time())
    ].copy()
    
    if len(df_vwap) == 0:
        print(f"{roll_date}: No trades in VWAP window")
        continue
    
    vwap = (df_vwap['trade_price'] * df_vwap['trade_size']).sum() / df_vwap['trade_size'].sum()
    
    results.append({
        'roll_date': roll_date,
        'next_expiry': next_expiry,
        'spx_level': round(spx_level, 2),
        'atm_strike': atm_strike,
        'vwap_trades': len(df_vwap),
        'vwap_volume': df_vwap['trade_size'].sum(),
        'vwap_price': round(vwap, 4)
    })
    print(f"{roll_date} ✓ Strike={atm_strike}, VWAP=${vwap:.4f}, Trades={len(df_vwap)}")

print("\n=== Done ===")
df_results = pd.DataFrame(results)
print(df_results)

2022-01-21 ✓ Strike=4480.0, VWAP=$92.8499, Trades=685
2022-02-18 ✓ Strike=4365.0, VWAP=$104.3596, Trades=946
2022-03-18 ✓ Strike=4410.0, VWAP=$98.4000, Trades=2
2022-04-14 ✓ Strike=4430.0, VWAP=$92.5078, Trades=759
2022-05-20 ✓ Strike=3870.0, VWAP=$115.0523, Trades=109
2022-06-17 ✓ Strike=3650.0, VWAP=$130.6748, Trades=13
2022-07-15 ✓ Strike=3850.0, VWAP=$107.0507, Trades=987
2022-08-19 ✓ Strike=4145.0, VWAP=$141.4500, Trades=1
2022-09-16 ✓ Strike=3860.0, VWAP=$114.8429, Trades=2
2022-10-21: No trades in VWAP window
2022-11-18 ✓ Strike=3955.0, VWAP=$95.6717, Trades=883
2022-12-16 ✓ Strike=3845.0, VWAP=$100.6423, Trades=8

=== Done ===
     roll_date next_expiry  spx_level  atm_strike  vwap_trades  vwap_volume  \
0   2022-01-21  2022-02-18    4478.28      4480.0          685         3738   
1   2022-02-18  2022-03-18    4364.73      4365.0          946         4276   
2   2022-03-18  2022-04-14    4409.05      4410.0            2           33   
3   2022-04-14  2022-05-20    4427.69    

In [12]:
# -- Diagnose Mar 18 Roll Date --

df_mar = pd.read_csv("../data/raw/cboe_spx_2022/UnderlyingOptionsTradesCalcs_2022-03-18.csv")
df_mar['quote_datetime'] = pd.to_datetime(df_mar['quote_datetime'])

df_mar_filtered = df_mar[
    (df_mar['root'] == 'SPX') &
    (df_mar['option_type'] == 'C') &
    (df_mar['quote_datetime'].dt.time >= pd.Timestamp('09:30').time()) &
    (df_mar['quote_datetime'].dt.time <= pd.Timestamp('16:15').time())
].copy()

print("Expiration dates on Mar 18:")
print(df_mar_filtered['expiration'].value_counts().head(10))

Expiration dates on Mar 18:
expiration
2022-04-14    3070
2022-06-17    1945
2022-05-20    1205
2023-12-15     764
2022-09-16     635
2022-07-15     383
2022-12-16     293
2023-01-20     267
2022-08-19     213
2023-06-16     156
Name: count, dtype: int64


In [11]:
# -- Diagnose ATM Strike Selection on Mar 18 --

df_apr14 = df_mar_filtered[df_mar_filtered['expiration'] == '2022-04-14'].copy()

# 11:00 AM window
window_11am = df_apr14[
    (df_apr14['quote_datetime'].dt.time >= pd.Timestamp('10:50').time()) &
    (df_apr14['quote_datetime'].dt.time <= pd.Timestamp('11:00').time())
]

print(f"Trades near 11:00 AM: {len(window_11am)}")
print(f"SPX level: {window_11am['underlying_bid'].mean():.2f}")

# Available strikes near ATM
spx_level = window_11am['underlying_bid'].mean()
available_strikes = sorted(df_apr14['strike'].unique())
near_atm = [s for s in available_strikes if abs(s - spx_level) < 100]
print(f"\nAvailable strikes near ATM: {near_atm}")

# VWAP window trades per strike
print("\nTrades in VWAP window per strike (near ATM):")
df_vwap_all = df_apr14[
    (df_apr14['quote_datetime'].dt.time >= pd.Timestamp('11:30').time()) &
    (df_apr14['quote_datetime'].dt.time <= pd.Timestamp('13:30').time()) &
    (df_apr14['strike'].isin(near_atm))
]
print(df_vwap_all.groupby('strike')['trade_size'].agg(['count', 'sum']))

Trades near 11:00 AM: 17
SPX level: 4409.05

Available strikes near ATM: [np.float64(4315.0), np.float64(4320.0), np.float64(4325.0), np.float64(4330.0), np.float64(4335.0), np.float64(4350.0), np.float64(4355.0), np.float64(4360.0), np.float64(4365.0), np.float64(4370.0), np.float64(4375.0), np.float64(4380.0), np.float64(4385.0), np.float64(4390.0), np.float64(4395.0), np.float64(4400.0), np.float64(4405.0), np.float64(4410.0), np.float64(4415.0), np.float64(4420.0), np.float64(4425.0), np.float64(4430.0), np.float64(4435.0), np.float64(4440.0), np.float64(4445.0), np.float64(4450.0), np.float64(4455.0), np.float64(4460.0), np.float64(4465.0), np.float64(4470.0), np.float64(4475.0), np.float64(4480.0), np.float64(4485.0), np.float64(4490.0), np.float64(4495.0), np.float64(4500.0), np.float64(4505.0)]

Trades in VWAP window per strike (near ATM):
        count   sum
strike             
4315.0      1     5
4325.0      1     1
4330.0      1     1
4350.0     11   529
4370.0      1     1


### Thought Process — ATM Strike Selection Issue

On Mar 18, SPX level = 4409.05 → selected ATM = 4410.0 (2 trades only).
But actual market volume concentrated at 4415.0 (422 trades).

Two possible explanations:
1. SPX moved slightly after 11:00 AM, making 4415.0 the de facto ATM
2. Our 10:50~11:00 average may not match BXM's exact "last reported value
   just before 11:00 AM" — BXM uses a single last tick, not an average

This suggests we need to refine the 11:00 AM SPX level calculation:
use the last single tick before 11:00 AM instead of a 10-minute average.

In [13]:
# -- Refined ATM Strike Selection (Last Tick Before 11:00 AM) --

roll_dates = [
    "2022-01-21", "2022-02-18", "2022-03-18", "2022-04-14",
    "2022-05-20", "2022-06-17", "2022-07-15", "2022-08-19",
    "2022-09-16", "2022-10-21", "2022-11-18", "2022-12-16"
]

results_v2 = []

for roll_date in roll_dates:
    path = f"../data/raw/cboe_spx_2022/UnderlyingOptionsTradesCalcs_{roll_date}.csv"
    
    if not os.path.exists(path):
        print(f"{roll_date}: File not found")
        continue
    
    df_r = pd.read_csv(path)
    df_r['quote_datetime'] = pd.to_datetime(df_r['quote_datetime'])
    
    # Filter: SPX monthly calls, regular session
    df_r = df_r[
        (df_r['root'] == 'SPX') &
        (df_r['option_type'] == 'C') &
        (df_r['quote_datetime'].dt.time >= pd.Timestamp('09:30').time()) &
        (df_r['quote_datetime'].dt.time <= pd.Timestamp('16:15').time())
    ].copy()
    
    # Next roll date expiry
    next_idx = roll_dates.index(roll_date) + 1
    next_expiry = roll_dates[next_idx] if next_idx < len(roll_dates) else "2023-01-20"
    df_next = df_r[df_r['expiration'] == next_expiry].copy()
    
    if len(df_next) == 0:
        print(f"{roll_date}: No data for expiry {next_expiry}")
        continue
    
    # Last single tick before 11:00 AM (refined)
    before_11am = df_next[
        df_next['quote_datetime'].dt.time < pd.Timestamp('11:00').time()
    ].copy()
    
    if len(before_11am) == 0:
        print(f"{roll_date}: No data before 11:00 AM")
        continue
    
    last_tick = before_11am.loc[before_11am['quote_datetime'].idxmax()]
    spx_level = last_tick['underlying_bid']
    
    # ATM Strike
    available_strikes = sorted(df_next['strike'].unique())
    valid_strikes = [s for s in available_strikes if s >= spx_level]
    
    if not valid_strikes:
        print(f"{roll_date}: No valid strike found")
        continue
    
    atm_strike = min(valid_strikes)
    
    # VWAP 11:30 AM ~ 1:30 PM ET
    df_vwap = df_next[
        (df_next['strike'] == atm_strike) &
        (df_next['quote_datetime'].dt.time >= pd.Timestamp('11:30').time()) &
        (df_next['quote_datetime'].dt.time <= pd.Timestamp('13:30').time())
    ].copy()
    
    if len(df_vwap) == 0:
        print(f"{roll_date}: No trades in VWAP window")
        continue
    
    vwap = (df_vwap['trade_price'] * df_vwap['trade_size']).sum() / df_vwap['trade_size'].sum()
    
    results_v2.append({
        'roll_date': roll_date,
        'next_expiry': next_expiry,
        'spx_level': round(spx_level, 2),
        'atm_strike': atm_strike,
        'vwap_trades': len(df_vwap),
        'vwap_volume': df_vwap['trade_size'].sum(),
        'vwap_price': round(vwap, 4)
    })
    print(f"{roll_date} ✓ Strike={atm_strike}, VWAP=${vwap:.4f}, Trades={len(df_vwap)}")

df_results_v2 = pd.DataFrame(results_v2)
print("\n=== V2 Results ===")
print(df_results_v2)

2022-01-21 ✓ Strike=4475.0, VWAP=$89.0573, Trades=14
2022-02-18 ✓ Strike=4365.0, VWAP=$104.3596, Trades=946
2022-03-18 ✓ Strike=4415.0, VWAP=$99.9869, Trades=422
2022-04-14 ✓ Strike=4430.0, VWAP=$92.5078, Trades=759
2022-05-20 ✓ Strike=3885.0, VWAP=$97.6424, Trades=864
2022-06-17 ✓ Strike=3655.0, VWAP=$122.0219, Trades=1526
2022-07-15 ✓ Strike=3850.0, VWAP=$107.0507, Trades=987
2022-08-19 ✓ Strike=4235.0, VWAP=$85.2898, Trades=923
2022-09-16 ✓ Strike=3855.0, VWAP=$118.5279, Trades=1062
2022-10-21 ✓ Strike=3680.0, VWAP=$135.6402, Trades=1195
2022-11-18 ✓ Strike=3950.0, VWAP=$96.3051, Trades=53
2022-12-16 ✓ Strike=3850.0, VWAP=$97.4802, Trades=1200

=== V2 Results ===
     roll_date next_expiry  spx_level  atm_strike  vwap_trades  vwap_volume  \
0   2022-01-21  2022-02-18    4471.83      4475.0           14         2001   
1   2022-02-18  2022-03-18    4363.49      4365.0          946         4276   
2   2022-03-18  2022-04-14    4410.46      4415.0          422         5086   
3   2022-

In [14]:
# -- Diagnose Jan 21 Low Trade Count --

df_jan21_diag = pd.read_csv("../data/raw/cboe_spx_2022/UnderlyingOptionsTradesCalcs_2022-01-21.csv")
df_jan21_diag['quote_datetime'] = pd.to_datetime(df_jan21_diag['quote_datetime'])

df_feb18 = df_jan21_diag[
    (df_jan21_diag['root'] == 'SPX') &
    (df_jan21_diag['option_type'] == 'C') &
    (df_jan21_diag['expiration'] == '2022-02-18') &
    (df_jan21_diag['quote_datetime'].dt.time >= pd.Timestamp('09:30').time()) &
    (df_jan21_diag['quote_datetime'].dt.time <= pd.Timestamp('16:15').time())
].copy()

# Last tick before 11:00 AM
before_11am = df_feb18[df_feb18['quote_datetime'].dt.time < pd.Timestamp('11:00').time()]
last_tick = before_11am.loc[before_11am['quote_datetime'].idxmax()]
spx_level = last_tick['underlying_bid']

print(f"SPX last tick before 11:00 AM: {spx_level:.2f}")
print(f"Last tick time: {last_tick['quote_datetime']}")

# VWAP window trades per strike near ATM
df_vwap_all = df_feb18[
    (df_feb18['quote_datetime'].dt.time >= pd.Timestamp('11:30').time()) &
    (df_feb18['quote_datetime'].dt.time <= pd.Timestamp('13:30').time())
]

near_atm = [s for s in sorted(df_feb18['strike'].unique()) if abs(s - spx_level) < 100]
print(f"\nTrades in VWAP window per strike (near ATM):")
print(df_vwap_all[df_vwap_all['strike'].isin(near_atm)].groupby('strike')['trade_size'].agg(['count','sum']))

SPX last tick before 11:00 AM: 4471.83
Last tick time: 2022-01-21 10:59:23.103000

Trades in VWAP window per strike (near ATM):
        count   sum
strike             
4375.0      1     3
4400.0     17    60
4440.0      1     1
4450.0     12  1127
4460.0      1    30
4470.0      1    30
4475.0     14  2001
4480.0    685  3738
4485.0      6   141
4490.0      7   225
4495.0      2     2
4500.0     29  1239
4505.0     12   685
4510.0      2   145
4515.0      2     2
4530.0      8    23
4550.0     13   546
4560.0      3     4
4565.0      2     2
4570.0     11    26


### Thought Process — Systematic ATM Strike Selection Bias

Recurring pattern across multiple roll dates:
- Our selected strike has very few trades in VWAP window
- Actual volume concentrates 1-2 strikes higher

Root cause: we use underlying_bid from the last options tick before 11:00 AM,
but SPX continues to move between 11:00 AM and 11:30 AM (VWAP window start).
By the time the VWAP window opens, the market has already repriced to a
higher ATM strike.

Potential fix: use SPX index level data (from Bloomberg SPX_Bloomberg_2022.csv)
at exactly 11:00 AM instead of options-derived underlying_bid.
However, our Bloomberg data is daily only — not intraday.

Open question for Professor Dodson:
What is the exact data source for the 11:00 AM SPX level in BXM methodology?